# PatientDoctorBridge — Gemma 4 on a free GPU (Colab)

This notebook runs **Ollama + Gemma 4 on a free Colab T4 GPU** and exposes it over the internet so your CPU laptop can use it as its inference backend.

**How it fits together:** your laptop keeps running the `pdb server` (audio capture, Whisper ASR, TTS, the web UI). Only the Gemma 4 LLM calls (translation, triage, OCR) are sent to this GPU. You point the laptop at this notebook by setting one environment variable, `PDB_OLLAMA_HOST`.

> **Important / competition note:** this is for *experiencing GPU speed and development only*. The Kaggle Gemma 4 Impact Challenge requires inference to run locally with nothing leaving the device, so a cloud GPU is **not** valid for your final privacy claim. For a compliant GPU run, put Ollama on a GPU machine on your own LAN instead.

### Steps
1. **Runtime → Change runtime type → Hardware accelerator: T4 GPU**, then run the cells top to bottom.
2. Create a free ngrok account (https://dashboard.ngrok.com) and copy your authtoken into the tunnel cell.
3. Copy the printed `PDB_OLLAMA_HOST` value onto your laptop (see the last cell).

## 1. Confirm a GPU is attached
If this errors or shows no GPU, fix the runtime type first (Runtime → Change runtime type → T4 GPU).

In [ ]:
!nvidia-smi

## 2. Install Ollama

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Start the Ollama server
We bind to `0.0.0.0` so the tunnel can reach it, and run it in the background.

In [ ]:
import os, subprocess, time, urllib.request

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
# Keep models resident so you don't pay a reload tax between requests.
os.environ['OLLAMA_KEEP_ALIVE'] = '30m'

subprocess.Popen(['ollama', 'serve'])

# Wait for the server to come up
for _ in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2)
        print('Ollama is up.')
        break
    except Exception:
        time.sleep(1)
else:
    print('Ollama did not start — re-run this cell.')

## 4. Pull the Gemma 4 models
Your project uses **only** `gemma4:e2b` (fast translation) and `gemma4:e4b` (reasoning / triage / OCR).

> If `ollama pull gemma4:e2b` reports the tag is not found in the public registry, obtain the models here **the same way you installed them on your laptop** (e.g. your custom Modelfile / `ollama create`), so the tags match exactly. Do not substitute a different model — the project requires the `gemma4:*` family only.

In [ ]:
!ollama pull gemma4:e2b
!ollama pull gemma4:e4b
!ollama list

## 5. Expose Ollama to your laptop via ngrok
Paste your ngrok authtoken below. The `host_header` rewrite is **required** — without it Ollama rejects tunnelled requests with a 403.

In [ ]:
!pip -q install pyngrok
from pyngrok import ngrok

NGROK_AUTHTOKEN = "PASTE_YOUR_NGROK_AUTHTOKEN_HERE"  # from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# host_header rewrite makes Ollama see Host: localhost:11434 and accept the request
tunnel = ngrok.connect(11434, "http", host_header="localhost:11434")
public_url = tunnel.public_url
if public_url.startswith("http://"):
    public_url = "https://" + public_url[len("http://"):]

print("\n=== Set this on your LAPTOP, then start `pdb server` ===\n")
print("  PDB_OLLAMA_HOST=" + public_url)
print("\nKeep this Colab tab open and the runtime alive while you use the app.")

## 6. (Optional) Smoke-test the GPU path from here
Confirms the model answers and that it's running on the GPU.

In [ ]:
import json, time, urllib.request

payload = {
    "model": "gemma4:e2b",
    "prompt": "Translate to English: Mujhe do din se tej bukhar hai.",
    "stream": False,
    "think": False,
    "options": {"num_predict": 64, "temperature": 0.2},
}
req = urllib.request.Request(
    "http://localhost:11434/api/generate",
    data=json.dumps(payload).encode(),
    headers={"Content-Type": "application/json"},
)
t0 = time.time()
resp = json.loads(urllib.request.urlopen(req, timeout=120).read())
print(f"Response in {time.time()-t0:.2f}s:\n", resp.get("response", "").strip())

## On your laptop

Open a terminal **in your project folder** and set the variable from cell 5, then start the server in the *same* terminal:

**Windows (Command Prompt):**
```
set PDB_OLLAMA_HOST=https://xxxx.ngrok-free.app
pdb server
```

**Windows (PowerShell):**
```
$env:PDB_OLLAMA_HOST = "https://xxxx.ngrok-free.app"
pdb server
```

**macOS / Linux:**
```
export PDB_OLLAMA_HOST="https://xxxx.ngrok-free.app"
pdb server
```

To go back to local CPU, just close that terminal (or unset the variable) and start `pdb server` normally.

### Things to expect
- **Only Gemma calls go to the GPU.** Whisper ASR and TTS still run on your laptop CPU, so you'll still feel some transcription latency — but translation, triage, and prescription OCR should be dramatically faster.
- **The ngrok URL changes every session.** Each time you restart this notebook you'll get a new URL and must update `PDB_OLLAMA_HOST`.
- **Colab free runtimes are ephemeral** (disconnect after idle, ~12h max), and the model pull repeats on each fresh session.
- **First request after startup** may still be slow while the model loads into GPU memory; after that `keep_alive` keeps it warm.